# 1. Coordenadas -- Soja no Parana

Pipeline paralelo ao de milho (arquivos `1.Coordinates.ipynb` ... `7. Optimization.ipynb`),
dedicado a estimativa de produtividade de **soja** para **todos os municipios do Parana**,
usando clima do **BR-DWGD** e produtividade do **IBGE/SIDRA**.

Diferente do pipeline de milho (que parte de uma grade irregular de pontos e faz um join
espacial com os municipios), aqui o ponto de partida ja e o municipio: usamos o
**centroide (representative point) de cada municipio do PR** como coordenada de simulacao.
Isso elimina a etapa de join espacial e garante 1 ponto = 1 municipio = 1 serie de
produtividade do IBGE.


In [1]:
import os

import geopandas as gpd
import pandas as pd

import sys
sys.path.append(os.path.join(os.getcwd(), 'util'))

from util.utils import obter_elevacao_multiplas_fontes
from util.utils_soja_pr import setup_paths_soja_pr

paths = setup_paths_soja_pr()
paths


Building PCSE demo database at: C:\Users\luis.mcosta\.pcse\pcse.db ... OK


{'BASE': 'd:\\_py\\AgroIA_prod',
 'DATA': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr',
 'COORDINATES': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr\\coordinates_pr.xlsx',
 'COMPLETO': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr\\completo',
 'AGRO': 'd:\\_py\\AgroIA_prod\\inputs\\data\\agro\\agro_soybean_pr.agro',
 'CROP': 'd:\\_py\\AgroIA_prod\\inputs\\data\\crop',
 'SOIL': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soil\\ec3.soil',
 'WEATHER_RAW': 'D:\\_py\\Clima_AgroIA\\data-raw\\xavier-data',
 'RESULTS': 'd:\\_py\\AgroIA_prod\\output\\soja_pr\\Sensitivity Analysis',
 'OPTIMIZATION': 'd:\\_py\\AgroIA_prod\\output\\soja_pr\\Optimization'}

## Baixar malha de municipios do Brasil e filtrar o Parana (UF 41)

Reusa a mesma fonte do pipeline de milho (`tbrugz/geodata-br`), mas filtra apenas os
municipios cujo codigo IBGE comeca com `41` (codigo da UF do Parana), em vez de usar a
malha do Brasil inteiro.


In [2]:
url_municipios = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"

print("Carregando mapa de municipios do Brasil...")
municipios_gdf = gpd.read_file(url_municipios)
municipios_gdf.rename(columns={'id': 'cod_municipio'}, inplace=True)

pr_gdf = municipios_gdf[municipios_gdf['cod_municipio'].astype(str).str.startswith('41')].copy()
pr_gdf.reset_index(drop=True, inplace=True)

print(f"Municipios do Parana encontrados: {len(pr_gdf)}")
pr_gdf.head()


Carregando mapa de municipios do Brasil...
Municipios do Parana encontrados: 399


,cod_municipio,name,description,geometry
0,4100103,Abatiá,Abatiá,"POLYGON ((-50.22434 -23.23695, -50.2146 -23.23..."
1,4100202,Adrianópolis,Adrianópolis,"POLYGON ((-49.03421 -24.63401, -49.0179 -24.64..."
2,4100301,Agudos do Sul,Agudos do Sul,"POLYGON ((-49.30516 -25.94478, -49.30352 -25.9..."
3,4100400,Almirante Tamandaré,Almirante Tamandaré,"POLYGON ((-49.25897 -25.23076, -49.25038 -25.2..."
4,4100459,Altamira do Paraná,Altamira do Paraná,"POLYGON ((-52.7795 -24.73458, -52.77761 -24.74..."


## Gerar 1 ponto por municipio (centroide dentro do poligono) e buscar elevacao

`representative_point()` garante um ponto sempre dentro do poligono (diferente de
`centroid`, que pode cair fora em municipios com geometria concava).


In [3]:
pr_gdf['geometry_point'] = pr_gdf.geometry.representative_point()
pr_gdf['lat'] = pr_gdf['geometry_point'].y
pr_gdf['lon'] = pr_gdf['geometry_point'].x

registros = []
for _, row in pr_gdf.iterrows():
    elevacao, fonte = obter_elevacao_multiplas_fontes(row['lat'], row['lon'])
    registros.append({
        'cod_municipio': str(row['cod_municipio']),
        'name': row['name'],
        'lat': row['lat'],
        'lon': row['lon'],
        'country': 'Brazil',
        'state': 'Parana',
        'elevation_m': elevacao,
    })
    print(f"{row['name']:30s} lat={row['lat']:.4f} lon={row['lon']:.4f} elev={elevacao} ({fonte})")

df_coords_pr = pd.DataFrame(registros)
df_coords_pr.shape


Abatiá                         lat=-23.3018 lon=-50.3387 elev=609.72 (NASA_POWER)
Adrianópolis                   lat=-24.7927 lon=-48.8007 elev=678.89 (NASA_POWER)
Agudos do Sul                  lat=-26.0403 lon=-49.3055 elev=849.62 (NASA_POWER)
Almirante Tamandaré            lat=-25.3021 lon=-49.3275 elev=905.85 (NASA_POWER)
Altamira do Paraná             lat=-24.8242 lon=-52.6713 elev=636.07 (NASA_POWER)
Altônia                        lat=-23.9121 lon=-53.9165 elev=315.06 (NASA_POWER)
Alto Paraná                    lat=-23.0737 lon=-52.3099 elev=425.62 (NASA_POWER)
Alto Piquiri                   lat=-24.0899 lon=-53.3932 elev=408.68 (NASA_POWER)
Alvorada do Sul                lat=-22.8178 lon=-51.2382 elev=454.93 (NASA_POWER)
Amaporã                        lat=-23.1386 lon=-52.8345 elev=328.34 (NASA_POWER)
Ampére                         lat=-25.9086 lon=-53.4932 elev=492.74 (NASA_POWER)
Anahy                          lat=-24.6391 lon=-53.1418 elev=445.62 (NASA_POWER)
Andirá          

(399, 7)

In [4]:
os.makedirs(os.path.dirname(paths['COORDINATES']), exist_ok=True)

with pd.ExcelWriter(paths['COORDINATES'], engine='openpyxl') as writer:
    df_coords_pr.to_excel(writer, sheet_name='SIDRA-ids', index=False)

print(f"Coordenadas dos municipios do PR salvas em {paths['COORDINATES']}")
df_coords_pr.head()


Coordenadas dos municipios do PR salvas em d:\_py\AgroIA_prod\inputs\data\soja_pr\coordinates_pr.xlsx


,cod_municipio,name,lat,lon,country,state,elevation_m
0,4100103,Abatiá,-23.301821,-50.338724,Brazil,Parana,609.72
1,4100202,Adrianópolis,-24.792651,-48.800739,Brazil,Parana,678.89
2,4100301,Agudos do Sul,-26.040265,-49.305534,Brazil,Parana,849.62
3,4100400,Almirante Tamandaré,-25.302081,-49.327528,Brazil,Parana,905.85
4,4100459,Altamira do Paraná,-24.824170,-52.671320,Brazil,Parana,636.07
